In [1]:
import sys
import os
import cv2


sys.path.append(os.path.abspath('..'))

import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# Import từ thư mục core
from core.utils import seed_everything, get_device, build_dataloaders, train_one_epoch, evaluate, count_parameters, plot_history
from core.models import BNN, CNN

In [2]:
# Configs
EPOCHS = 7
BATCH_SIZE = 32
LR = 1e-3
VAL_RATIO = 0.1
SEED = 42
MODEL_NAME = "BNN"
DATASET = "mnist" 
OUT_DIR = Path("runs_"+DATASET+"_"+MODEL_NAME)
OUT_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)
device = get_device()
print(f"Sử dụng thiết bị: {device}")

Sử dụng thiết bị: cpu


In [3]:
train_loader, val_loader, test_loader = build_dataloaders(
    BATCH_SIZE, 
    VAL_RATIO, 
    SEED, 
    binarize_input=False, 
    dataset=DATASET #nhị phân hóa dữ liệu, chọn dataset
)

In [4]:
#
model = BNN(activation_type="binary").to(device) #chọn model và activation
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [5]:
print(f"Tổng tham số {MODEL_NAME}: {count_parameters(model):,}")
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = -1.0
best_state = None

Tổng tham số BNN: 54,474


In [6]:
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train: {train_acc*100:.2f}% | Val: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

if best_state is not None:
    torch.save(best_state, OUT_DIR / "best_model.pt")
    model.load_state_dict(best_state)

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print("-" * 40)
print(f"Test accuracy: {test_acc*100:.2f}%")

Epoch 01/07 | Train: 88.05% | Val: 83.97%
Epoch 02/07 | Train: 90.53% | Val: 91.43%
Epoch 03/07 | Train: 92.77% | Val: 92.12%
Epoch 04/07 | Train: 93.15% | Val: 94.87%
Epoch 05/07 | Train: 93.37% | Val: 91.30%
Epoch 06/07 | Train: 93.83% | Val: 93.85%
Epoch 07/07 | Train: 93.91% | Val: 94.02%
----------------------------------------
Test accuracy: 94.08%


In [7]:
#vẽ biểu đồ: chọn tên biểu đồ, tên file
plot_history(history, OUT_DIR, DATASET+"_"+MODEL_NAME, file_name=DATASET+"_"+MODEL_NAME+"_history")

np.save(OUT_DIR / "train_loss.npy", np.array(history["train_loss"]))
np.save(OUT_DIR / "train_acc.npy", np.array(history["train_acc"]))
np.save(OUT_DIR / "val_loss.npy", np.array(history["val_loss"]))
np.save(OUT_DIR / "val_acc.npy", np.array(history["val_acc"]))